## 6.1 TRPO
我们在上一章的结尾得到了一个重要结论：**重要性采样解决了On-policy的数据效率问题，但引入了更严重的方差爆炸问题**。

当新旧策略差异较大时，重要性权重$\rho_t = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$会变得极大或极小，导致梯度更新的幅度完全失控。一次错误的更新就可能让策略彻底崩溃，再也无法恢复。

这就引出了一个核心问题：**我们到底能允许新旧策略之间有多大的差异？**

1995年提出的**信任域策略优化（Trust Region Policy Optimization, TRPO）** 第一次从理论上严格回答了这个问题。我们每次只能对策略做 **足够小的更新**，保证新策略$\pi_\theta$和旧策略$\pi_{\theta_{old}}$之间的差异不超过一个极小的阈值。只要差异足够小，重要性采样的近似就是有效的，梯度估计的方差就是可控的。

TRPO用 **KL散度** 来衡量两个策略之间的差异，并提出了一个带约束的优化目标：
$$
\begin{align*}
\max_\theta \quad & \mathbb{E}_{t} \left[ \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)} \hat{A}_t \right] \\
\text{s.t.} \quad & \mathbb{E}_{t} \left[ D_{KL}(\pi_\theta(\cdot|s_t) || \pi_{\theta_{old}}(\cdot|s_t)) \right] \leq \delta
\end{align*}
$$

其中$\delta$是一个极小的常数（通常取0.01），代表我们允许的最大策略差异。

这个约束就是著名的 **信任域（Trust Region）**：我们只在旧策略附近的一个小区域内信任我们的梯度估计，超出这个区域，我们就认为梯度是不可靠的，不允许更新。

然而，TRPO需要求解一个带约束的优化问题，这要求计算 **二阶导数（Hessian矩阵）**，计算量是参数数量的平方级。对于一个1.5B参数的模型，Hessian矩阵的大小是$1.5B \times 1.5B$，这在任何现有硬件上都是不可能存储和计算的。

### 6.1.1 PPO
PPO没有使用复杂的带约束优化，而是直接把目标函数改成了：
$$
L^{CLIP}(\theta) = \mathbb{E}_{t} \left[ \min \left( r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]
$$

这个式子的本质是：
- 如果$r_t(\theta)$在$[1-\epsilon, 1+\epsilon]$之间，就正常使用重要性采样的梯度
- 如果$r_t(\theta)$超出了这个范围，就把它截断到边界上，不再允许进一步增大或减小

这就用一个简单的算术操作，实现了和TRPO的KL约束几乎完全相同的效果：**把策略更新限制在了旧策略附近的一个小区域内**。更具体的可以看[隔壁引言一开始的介绍](../post-train/post-train.ipynb)。

### 6.1.2 GRPO
我们已经走完了策略梯度的完整演进路径：
1. **REINFORCE**：最基础的On-policy策略梯度，无偏但方差极高
2. **REINFORCE with Baseline**：引入价值网络作为基线，零偏差降低方差
3. **重要性采样**：将On-policy梯度转换为Off-policy，提升数据效率
4. **PPO**：加入Clip截断机制，限制策略更新幅度，解决重要性采样的方差爆炸问题

然而，PPO解决了策略梯度的稳定性和数据效率问题，成为了通用RLHF的标准算法。但我们在引言中已经分析过，PPO天生是为**主观偏好对齐**设计的，在**客观可验证的推理任务**中存在一个巨大的冗余：
> PPO需要一个独立的Critic（价值网络）来估计状态价值$V(s_t)$，但在推理任务中，这个Critic是完全多余的。

这就是GRPO（Group Relative Policy Optimization）诞生的核心动机：**用"同题多答"的组内相对奖励，完全替代Critic网络，在保留PPO全部稳定性的同时，将训练显存开销减半**。
| 特性 | PPO | GRPO |
|------|-----|------|
| 所需模型 | 4个（Policy、Reference、RM、Critic） | 2个（Policy、Reference） |
| 基线估计 | 训练Critic网络估计$V(s_t)$ | 同题多答的组内平均奖励 |
| 显存开销 | 极高（需要同时加载4个大模型） | 极低（仅需加载1个训练模型） |
| 工程复杂度 | 极高（需要同步4个模型的更新） | 极低（和SFT几乎一致） |
| 适用场景 | 主观偏好对齐（对话、安全） | 客观可验证任务（数学、代码） |
| 训练稳定性 | 中等（容易出现KL爆炸） | 极好（无Critic过拟合问题） |

PPO用Critic网络拟合$V(s_t)$，本质上是在估计"从状态$s_t$出发的期望回报"。而如果我们对同一个问题采样$G$条回答，那么这$G$条回答的平均奖励，就是这个问题初始状态$s_0$的期望回报的**无偏蒙特卡洛估计**。

对于从问题分布采样得到的一个问题$q$，我们独立采样$G$条回答：
$$
\{o^{(1)}, o^{(2)}, \dots, o^{(G)}\}
$$

每条回答交给程序化奖励函数$R$打分，得到原始奖励：
$$
r^{(i)} = R(q, o^{(i)})
$$

GRPO将第$i$条回答的优势定义为它在该组内的**Z-score**：
$$
A^{(i)} = \frac{r^{(i)} - \mu}{\sigma + \epsilon} \tag{1}
$$

其中：
- $\mu = \frac{1}{G}\sum_{j=1}^G r^{(j)}$：组内奖励均值，充当基线$b(s)$
- $\sigma = \sqrt{\frac{1}{G}\sum_{j=1}^G (r^{(j)} - \mu)^2}$：组内奖励标准差
- $\epsilon = 10^{-8}$：防止除零的小常数

组归一化带来了两个又是：
1. **自动消除题目难度偏移**
   简单题大家都得1分，难题大家都得0分。如果直接用原始奖励$r^{(i)}$作为权重，模型会只学到简单题的模式，完全忽略难题。减去组均值后，模型学到的是"在同一道题中比平均水平好多少"，无论题目难易，梯度贡献都是均衡的。

2. **统一梯度尺度**
   除以标准差后，所有问题的优势值都被缩放到可比的范围内，消除了奖励量纲的影响。这使得不同题目、不同batch之间的梯度量级保持一致，训练更加稳定。

> BTW 还有一种简化版：只减均值，不除标准差
> 当组内 reward 的标准差很小，除以 std 会把噪声和极端样本不必要地放大
> 原始 GRPO 的标准差归一化 + 长度平均，会共同造成优化偏差，推动回复变长；Dr. GRPO 则主打去掉这种偏置，提高 token efficiency。TRL 文档也已经把 dr_grpo 收进 trainer，明确建议用常数而不是序列长度做分母来消除响应长度偏置。
## 6.2 GRPO-Clip
原始DeepSeekMath论文中的GRPO只提出了组优势估计，但现代开源实现实际上是**GRPO的组优势 + PPO的Clipped Surrogate Objective**的结合体，称为**GRPO-Clip**。
这是因为组优势只解决了基线估计的问题，没有解决重要性采样带来的方差爆炸问题。我们仍然需要PPO的Clip机制来限制策略更新幅度。
### 6.2.1 逐token概率比率
和PPO完全一致，我们用旧策略$\pi_{\theta_{old}}$采样数据后，需要计算第i条回答的第t个token，逐token的概率比率来修正分布偏移：
$$
\rho_{i,t}(\theta) = \frac{\pi_\theta(o_t^{(i)} \mid q, o_{<t}^{(i)})}{\pi_{\theta_{old}}(o_t^{(i)} \mid q, o_{<t}^{(i)})}
$$
这个比率是**按token计算**的，因为对于同一个token位置，新策略赋予它的概率可能和旧策略不同。所有token的$\rho_{i,t}$共同描述了整条轨迹的分布变化。

### 6.2.2 目标函数
GRPO-Clip的优化目标是最大化如下期望：
$$
\mathcal{J}_{GRPO}(\theta) = \mathbb{E}_{q, \{o^{(i)}\}\sim \pi_{\theta_{old}}} \left[ \frac{1}{G} \sum_{i=1}^G \frac{1}{|o^{(i)}|} \sum_{t=1}^{|o^{(i)}|} \min \left( \rho_{i,t} A^{(i)},\; \text{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon) A^{(i)} \right) \right] \tag{2}
$$
- $\frac{1}{G}\sum_{i=1}^G$：对同一问题的$G$条回答求平均
- $\frac{1}{|o^{(i)}|}\sum_{t=1}^{|o^{(i)}|}$：对每条回答内的所有token求平均（`masked_mean`）
- $\min(\cdot, \cdot)$：PPO的悲观下界构造，取两项中的较小值
- $A^{(i)}$：整条回答的组优势，**所有token共享同一个值**
- $\text{clip}(\rho_{i,t}, 1-\epsilon, 1+\epsilon)$：将比率限制在$[1-\epsilon, 1+\epsilon]$区间内，通常$\epsilon=0.2$
### 6.2.3 为什么所有token共享同一个优势
在数学推理任务中，奖励信号本身就是**序列级**的。我们只能判断"整个回答是否正确"，无法自动判断"推理链中的哪一步是对的，哪一步是错的"。

因此，$A^{(i)}$只能反映整条回复相对于组平均的优劣。这确实是GRPO的一个根本局限，但也是它能够去掉Critic的必要代价。后续的Process Reward Model（PRM）正是试图解决这个问题，提供token-level的奖励信号。
### 6.2.4 Clip 机制
通过构造一个悲观下界，确保策略更新不会超出"信任域"。
当$A^{(i)}>0$（这条回答优于组平均），我们想要增加这些token的概率，即让$\rho>1$。
- 当$\rho \in [1, 1+\epsilon]$：正常奖励，目标函数随$\rho$线性增长
- 当$\rho > 1+\epsilon$：目标函数被截断在$(1+\epsilon)A$，再增大$\rho$不会带来任何额外收益

**物理意义**：即使这个token看起来非常好，也不允许一次更新使其概率增长超过旧策略的20%。这给策略更新加了一个"速度上限"，防止模型因为单条幸运的正确回答而过度更新。

当$A^{(i)}<0$（这条回答比组平均差），我们想要降低这些token的概率，即让$\rho<1$。
- 当$\rho \in [1-\epsilon, 1]$：正常惩罚，目标函数随$\rho$减小而减小
- 当$\rho < 1-\epsilon$：目标函数被截断在$(1-\epsilon)A$，再减小$\rho$不会带来任何额外惩罚

**物理意义**：即使这个token看起来非常差，也不允许一次更新使其概率降低超过旧策略的20%。这防止了模型永久扼杀某些潜在的合理推理路径。
在代码中，我们通常会计算`clip_fraction`：
```python
clipped_mask = (surr2 < surr1)  # surr1=未裁剪, surr2=裁剪后
clip_fraction = clipped_mask.float().mean()
```

它衡量的是"有多少比例的token其概率比率已经越过了信任区间"：
- `clip_fraction > 0.3`：策略正在频繁"撞墙"，学习率过大或同一批数据训练太多遍
- `clip_fraction ≈ 0`：策略更新过于保守，学习率过小或$\epsilon$过大
## 6.3 长度归一化
### 6.3.1 `masked_mean`：样本平权（原始GRPO）
对于每条回答，先求自己内部的平均loss，再对所有回答求平均：
$$
L_{\text{batch}} = \frac{1}{G}\sum_{i=1}^G \left( \frac{1}{|o^{(i)}|} \sum_{t=1}^{|o^{(i)}|} \ell_{i,t} \right)
$$

**问题**：长回答中每个token的权重被稀释了。
- 短回答：每个 token 比较贵
- 长回答：每个 token 比较便宜
一条1000token的错误回答，每个token的惩罚只有一条10token错误回答的1/100。这会导致模型倾向于生成更长更冗余的序列，通过"堆长度"来碰运气。
### 6.3.2 `masked_normalize`：Token平权（Dr. GRPO）
固定一个全局常数$C$（通常是最大序列长度`max_seq_len`或batch内平均长度），所有token的loss都除以这个相同的常数：
$$
L_{\text{batch}} = \frac{1}{G \cdot C} \sum_{i=1}^G \sum_{t=1}^{|o^{(i)}|} \ell_{i,t}
$$

**优势**：每个token的权重一致，与所在回复的长度无关。长回答中的每一个正确步骤都能获得应有的奖励，而错误长回答中的每个冗余token也会受到足够的惩罚。这会自然抑制错误的过度延长，同时鼓励正确的深入推理。

Understanding R1-Zero-Like Training 明确指出，原始 GRPO 的优化形式存在 response length bias，会人为拉长回答，尤其是错误回答。为解决这个偏差，他们提出 Dr. GRPO；TRL 文档现在也明确写了：要用常数而不是序列长度作分母来消除该偏差。DAPO 论文进一步指出，在 RL 训练里，长序列与截断样本的处理会显著影响训练稳定性；它特别强调了：
- Token-Level Policy Gradient Loss：对长 CoT 场景很关键
- Overlong Reward Shaping / Filtering：能减少 reward noise，稳定训练

继续沿用的方法包括：
- init_vllm
- load_policy_into_vllm_instance
- tokenize_prompt_and_output
- compute_entropy
- get_response_log_probs
- r1_zero_reward_fn
- log_generations

In [ ]:
import os
import re
import math
import json
import random
from fractions import Fraction
from typing import List, Dict, Optional, Any, Callable, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import wandb

from tqdm.auto import tqdm
from torch.optim import AdamW
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizer,
    PreTrainedModel,
)

from vllm import LLM, SamplingParams
from unittest.mock import patch

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)

print("DEVICE =", DEVICE)

THINK_OPEN = "<think>"
THINK_CLOSE = "</think>"
ANSWER_OPEN = "<answer>"
ANSWER_CLOSE = "</answer>"

### 6.4.1 masked_mean / masked_normalize / 统一 objective 聚合器
“sequence-level vs token-level” 的代码落点

In [ ]:
def masked_mean(tensor: torch.Tensor, mask: torch.Tensor, dim: int = -1, eps: float = 1e-8):
    """
    sequence-level:
      先对每条样本按自身长度取平均，再在 batch 维度上聚合。
    """
    mask = mask.float()
    masked_sum = torch.sum(tensor * mask, dim=dim)
    mask_count = torch.sum(mask, dim=dim).clamp_min(eps)
    return masked_sum / mask_count


def masked_normalize(
    tensor: torch.Tensor,
    mask: torch.Tensor,
    normalize_constant: Optional[float] = None,
    eps: float = 1e-8,
):
    """
    token-level:
      对所有有效 token 求和，再除以一个全局常数。
    """
    mask = mask.float()
    masked_sum = torch.sum(tensor * mask)

    if normalize_constant is None:
        normalize_constant = mask.sum().clamp_min(eps)
    elif not torch.is_tensor(normalize_constant):
        normalize_constant = torch.tensor(
            float(normalize_constant),
            device=tensor.device,
            dtype=tensor.dtype,
        )

    return masked_sum / normalize_constant


def reduce_grpo_objective(
    per_token_objective: torch.Tensor,
    response_mask: torch.Tensor,
    length_norm_type: str = "mask_normalize",
    normalize_constant: Optional[float] = None,
):
    """
    将 per-token objective 聚合成一个标量 objective。
    支持：
      - mask_mean / masked_mean / sequence_level
      - mask_normalize / mask_norm / token_level
    """
    if length_norm_type in ["mask_mean", "masked_mean", "sequence_level"]:
        per_seq = masked_mean(per_token_objective, response_mask, dim=1)
        if normalize_constant is None:
            return per_seq.mean()
        else:
            return per_seq.sum() / float(normalize_constant)

    elif length_norm_type in ["mask_normalize", "mask_norm", "token_level"]:
        return masked_normalize(
            per_token_objective,
            response_mask,
            normalize_constant=normalize_constant,
        )

    else:
        raise ValueError(f"未知 length_norm_type: {length_norm_type}")

### 6.4.2 GRPO 组归一化优势函数

In [ ]:
def compute_group_normalized_rewards(
    reward_fn: Callable[[str, str], Dict[str, float]],
    rollout_responses: List[str],
    repeated_ground_truths: List[str],
    group_size: int,
    advantage_eps: float = 1e-8,
    normalize_by_std: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, Dict[str, float]]:
    """
    计算 GRPO 的组归一化优势。
    """
    assert len(rollout_responses) == len(repeated_ground_truths), "responses / ground_truths 数量必须一致"
    assert len(rollout_responses) % group_size == 0, "总样本数必须能被 group_size 整除"

    raw_rewards_list = []
    for response, truth in zip(rollout_responses, repeated_ground_truths):
        score_dict = reward_fn(response, truth)
        raw_rewards_list.append(score_dict["reward"])

    raw_rewards = torch.tensor(raw_rewards_list, dtype=torch.float32)

    num_questions = raw_rewards.shape[0] // group_size
    grouped_rewards = raw_rewards.view(num_questions, group_size)

    group_means = grouped_rewards.mean(dim=1, keepdim=True)

    if normalize_by_std:
        group_stds = grouped_rewards.std(dim=1, keepdim=True, unbiased=False)
        advantages = (grouped_rewards - group_means) / (group_stds + advantage_eps)
    else:
        advantages = grouped_rewards - group_means

    advantages = advantages.view(-1)

    metadata = {
        "mean_reward": raw_rewards.mean().item(),
        "std_reward": raw_rewards.std(unbiased=False).item(),
        "max_reward": raw_rewards.max().item(),
        "min_reward": raw_rewards.min().item(),
        "mean_advantage": advantages.mean().item(),
    }

    return advantages, raw_rewards, metadata

### 6.4.3 把 rollout 展平成扁平列表
vLLM 按“问题”为单位返回，每个问题里有 G 个 candidate。
训练前需要展平成：
- flat_prompts
- flat_responses
- repeated_ground_truths

In [ ]:
def flatten_group_rollout_outputs(
    batch_questions: List[Dict[str, str]],
    outputs,
) -> Tuple[List[str], List[str], List[str]]:
    """
    将 vLLM 的 group rollout 结果展平。
    返回：
      flat_prompts
      flat_responses
      repeated_ground_truths
    """
    flat_prompts = []
    flat_responses = []
    repeated_ground_truths = []

    for q_item, out in zip(batch_questions, outputs):
        for candidate in out.outputs:
            flat_prompts.append(q_item["prompt"])
            flat_responses.append(candidate.text)
            repeated_ground_truths.append(q_item["gold"])

    return flat_prompts, flat_responses, repeated_ground_truths

### 6.4.4 预计算 old policy 的 token log-probs
这是 GRPO / PPO 类算法最关键的“旧策略快照”之一。

不需要真的复制一份大模型，只要在 rollout 后、inner update 前，把当前 policy 对这批样本的 log-prob 算出来并缓存即可。

In [ ]:
@torch.no_grad()
def precompute_old_log_probs(
    policy: PreTrainedModel,
    tokenized_rollout_data: Dict[str, torch.Tensor],
    micro_batch_size: int,
    device: str,
) -> torch.Tensor:
    """
    用当前 policy（此时即 old policy）预计算 rollout 数据上的 token log-probs。
    返回：
      old_log_probs: [N, L]
    """
    policy.eval()

    all_old_log_probs = []
    total_samples = tokenized_rollout_data["input_ids"].size(0)

    for start in range(0, total_samples, micro_batch_size):
        end = min(start + micro_batch_size, total_samples)

        batch = {
            "input_ids": tokenized_rollout_data["input_ids"][start:end].to(device),
            "labels": tokenized_rollout_data["labels"][start:end].to(device),
            "attention_mask": tokenized_rollout_data["attention_mask"][start:end].to(device),
        }

        outputs = get_response_log_probs(
            model=policy,
            input_ids=batch["input_ids"],
            labels=batch["labels"],
            attention_mask=batch["attention_mask"],
            return_token_entropy=False,
        )

        all_old_log_probs.append(outputs["log_probs"].cpu())

    policy.train()
    old_log_probs = torch.cat(all_old_log_probs, dim=0)
    return old_log_probs

### 6.4.5 把 rollout 样本打包成训练数据集

In [ ]:
def prepare_rollout_training_dataset(
    flat_prompts: List[str],
    flat_responses: List[str],
    advantages: torch.Tensor,
    raw_rewards: torch.Tensor,
    tokenizer: PreTrainedTokenizer,
    policy: PreTrainedModel,
    micro_batch_size: int,
    device: str,
):
    """
    将 rollout 数据打包成 GRPO 训练集。
    返回：
      input_ids, labels, response_mask, valid_mask, attention_mask,
      raw_rewards, advantages, old_log_probs
    """
    tokenized_rollout_data = tokenize_prompt_and_output(
        prompt_strs=flat_prompts,
        output_strs=flat_responses,
        tokenizer=tokenizer,
    )

    old_log_probs = precompute_old_log_probs(
        policy=policy,
        tokenized_rollout_data=tokenized_rollout_data,
        micro_batch_size=micro_batch_size,
        device=device,
    )

    rollout_dataset = {
        **tokenized_rollout_data,
        "raw_rewards": raw_rewards.float().cpu(),
        "advantages": advantages.float().cpu(),
        "old_log_probs": old_log_probs.float().cpu(),
    }

    return rollout_dataset

### 6.4.6 切 logical batch / slice tensor dict

In [ ]:
def iterate_logical_batches(
    total_samples: int,
    logical_batch_size: int,
    shuffle: bool = True,
):
    """
    迭代产生 logical batch 的索引。
    """
    if shuffle:
        indices = torch.randperm(total_samples)
    else:
        indices = torch.arange(total_samples)

    for start in range(0, total_samples, logical_batch_size):
        yield indices[start:start + logical_batch_size]


def slice_tensor_dict(data: Dict[str, torch.Tensor], indices: torch.Tensor) -> Dict[str, torch.Tensor]:
    """
    按 indices 切一个 batch（仍保留在 CPU）。
    """
    return {
        k: v[indices]
        for k, v in data.items()
    }

### 6.4.7 计算三种 PG / 两种 GRPO 损失

In [ ]:
def compute_grpo_loss(
    current_token_log_probs: torch.Tensor,   # [B, L]
    response_mask: torch.Tensor,             # [B, L]
    advantages: torch.Tensor,                # [B]
    raw_rewards: torch.Tensor,               # [B]
    old_token_log_probs: Optional[torch.Tensor] = None,  # [B, L]
    loss_type: str = "grpo_clip",
    clip_eps: float = 0.2,
    length_norm_type: str = "mask_normalize",
    normalize_constant: Optional[float] = None,
):
    """
    统一计算：
      - no_baseline
      - reinforce_with_baseline
      - grpo_no_clip
      - grpo_clip
    """
    B, L = current_token_log_probs.shape
    mask = response_mask.float()
    valid_count = mask.sum().clamp_min(1.0)

    adv_token = advantages.unsqueeze(1).expand(B, L)
    reward_token = raw_rewards.unsqueeze(1).expand(B, L)

    ratio = None
    clip_fraction = torch.tensor(0.0, device=current_token_log_probs.device)
    ratio_mean = torch.tensor(1.0, device=current_token_log_probs.device)

    if loss_type == "no_baseline":
        per_token_objective = current_token_log_probs * reward_token

    elif loss_type == "reinforce_with_baseline":
        per_token_objective = current_token_log_probs * adv_token

    elif loss_type == "grpo_no_clip":
        if old_token_log_probs is None:
            raise ValueError("grpo_no_clip 需要 old_token_log_probs")
        ratio = torch.exp(current_token_log_probs - old_token_log_probs)
        ratio_mean = ((ratio * mask).sum() / valid_count).detach()
        per_token_objective = ratio * adv_token

    elif loss_type == "grpo_clip":
        if old_token_log_probs is None:
            raise ValueError("grpo_clip 需要 old_token_log_probs")
        ratio = torch.exp(current_token_log_probs - old_token_log_probs)
        clipped_ratio = ratio.clamp(1.0 - clip_eps, 1.0 + clip_eps)

        surr1 = ratio * adv_token
        surr2 = clipped_ratio * adv_token

        per_token_objective = torch.minimum(surr1, surr2)

        clipped_mask = ((surr2 < surr1).float() * mask)
        clip_fraction = (clipped_mask.sum() / valid_count).detach()
        ratio_mean = ((ratio * mask).sum() / valid_count).detach()

    else:
        raise ValueError(f"未知 loss_type: {loss_type}")

    objective = reduce_grpo_objective(
        per_token_objective=per_token_objective,
        response_mask=response_mask,
        length_norm_type=length_norm_type,
        normalize_constant=normalize_constant,
    )
    loss = -objective

    stats = {
        "objective": objective.detach().item(),
        "loss": loss.detach().item(),
        "clip_fraction": float(clip_fraction.item()),
        "ratio_mean": float(ratio_mean.item()),
    }
    return loss, stats

### 6.4.8 训练主循环需要的 rollout / eval / save helper

In [ ]:
def init_grpo_experiment(
    train_batch_size,
    micro_batch_size,
    wandb_project,
    wandb_run_name,
    prompt_path,
    extra_config=None,
):
    """
    初始化 wandb，加载 prompt 模板，计算 grad_accum_steps。
    """
    if train_batch_size % micro_batch_size != 0:
        raise ValueError("train_batch_size 必须能被 micro_batch_size 整除。")

    grad_accum_steps = train_batch_size // micro_batch_size

    config = dict(extra_config or {})
    config["grad_accum_steps"] = grad_accum_steps

    run = wandb.init(
        project=wandb_project,
        name=wandb_run_name,
        config=config,
    )

    run.define_metric("global_step")
    run.define_metric("grpo_step")
    run.define_metric("train/*", step_metric="global_step")
    run.define_metric("eval/*", step_metric="global_step")
    run.define_metric("rollout/*", step_metric="grpo_step")

    with open(prompt_path, "r") as f:
        r1_template = f.read().strip()

    return grad_accum_steps, run, r1_template

In [ ]:
def load_grpo_models_and_optimizer(model_id, device, vllm_device, seed, vllm_gpu_util, lr):
    """
    加载 tokenizer, policy, optimizer, vLLM。
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    policy = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        attn_implementation="flash_attention_2",
    ).to(device)

    policy.config.use_cache = False
    policy.gradient_checkpointing_enable()
    policy.train()

    optimizer = AdamW(policy.parameters(), lr=lr)
    vllm_inst = init_vllm(model_id, vllm_device, seed, vllm_gpu_util)

    return tokenizer, policy, optimizer, vllm_inst

### 6.4.9 准备训练问题池与验证集

In [ ]:
def prepare_question_pool(train_data_path, prompt_template):
    """
    构造训练问题池：
      [{"prompt": ..., "gold": ...}, ...]
    """
    question_pool = []

    with open(train_data_path, "r") as f:
        for line in f:
            item = json.loads(line)

            question = item["question"]
            if "gold" in item:
                gold = item["gold"]
            else:
                raw_a = item["answer"]
                gold = raw_a.split("####")[-1].strip() if "####" in raw_a else raw_a.strip()

            prompt = prompt_template.replace("{question}", question)
            question_pool.append({
                "prompt": prompt,
                "gold": gold,
            })

    return question_pool


def prepare_validation_samples(val_data_path, prompt_template, max_eval_samples):
    """
    构造验证集：
      [{"prompt": ..., "gold": ...}, ...]
    """
    val_samples = []

    with open(val_data_path, "r") as f:
        for i, line in enumerate(f):
            if i >= max_eval_samples:
                break

            item = json.loads(line)
            raw_a = item["answer"]
            gold = raw_a.split("####")[-1].strip() if "####" in raw_a else raw_a.strip()

            prompt = prompt_template.replace("{question}", item["question"])
            val_samples.append({
                "prompt": prompt,
                "gold": gold,
            })

    return val_samples

### 6.4.10 构造 eval / rollout sampling params


In [ ]:
def build_grpo_sampling_params(
    sampling_temperature: float,
    sampling_max_tokens: int,
    sampling_min_tokens: int,
    group_size: int,
):
    """
    构造：
      - eval_params：greedy
      - rollout_params：group sampling
    """
    eval_params = SamplingParams(
        temperature=0.0,
        max_tokens=sampling_max_tokens,
        stop=[ANSWER_CLOSE],
        include_stop_str_in_output=True,
    )

    rollout_params = SamplingParams(
        n=group_size,
        temperature=sampling_temperature,
        max_tokens=sampling_max_tokens,
        min_tokens=sampling_min_tokens,
        stop=[ANSWER_CLOSE],
        include_stop_str_in_output=True,
    )

    return eval_params, rollout_params

### 6.4.11 Step0 初始评估

In [ ]:
def run_initial_grpo_evaluation(policy, vllm_inst, eval_params, val_samples):
    """
    训练开始前的 baseline 评估。
    """
    print("\n[Step 0] 执行训练前初始评估...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_params,
        prompts=[s["prompt"] for s in val_samples],
        ground_truths=[s["gold"] for s in val_samples],
        reward_fn=r1_zero_reward_fn,
        step=0,
        log_prefix="eval",
    )
    print(f"Initial Accuracy: {metrics.get('eval/accuracy', 0):.2%}")

    policy.train()
    return metrics

### 6.4.12 抽一个 rollout question batch
- rollout_batch_size 是总回复数
- group_size 是每题生成几个回复
所以每轮实际抽题数是 `rollout_batch_size // group_size`

In [ ]:
def sample_rollout_questions(
    question_pool,
    rollout_batch_size: int,
    group_size: int,
):
    """
    从 question_pool 中抽取本轮 rollout 的题目。
    """
    if rollout_batch_size % group_size != 0:
        raise ValueError("rollout_batch_size 必须能被 group_size 整除。")

    num_questions = rollout_batch_size // group_size

    if num_questions <= len(question_pool):
        batch_questions = random.sample(question_pool, num_questions)
    else:
        batch_questions = random.choices(question_pool, k=num_questions)

    return batch_questions

### 6.4.13 外层循环数据准备器
执行一轮 rollout + 奖励计算 + 优势估计 + 打包训练数据

In [ ]:
def run_one_grpo_rollout(
    policy,
    vllm_inst,
    tokenizer,
    batch_questions,
    rollout_params,
    micro_batch_size,
    device,
    reward_fn,
    group_size,
    advantage_eps,
    normalize_by_std,
):
    """
    一轮 GRPO rollout:
      1. sync 权重到 vLLM
      2. group rollout
      3. flatten
      4. compute rewards / advantages
      5. tokenize
      6. precompute old log probs
      7. return rollout_dataset + metadata
    """
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    outputs = vllm_inst.generate(
        [q["prompt"] for q in batch_questions],
        rollout_params,
    )

    flat_prompts, flat_responses, repeated_ground_truths = flatten_group_rollout_outputs(
        batch_questions=batch_questions,
        outputs=outputs,
    )

    advantages, raw_rewards, reward_metadata = compute_group_normalized_rewards(
        reward_fn=reward_fn,
        rollout_responses=flat_responses,
        repeated_ground_truths=repeated_ground_truths,
        group_size=group_size,
        advantage_eps=advantage_eps,
        normalize_by_std=normalize_by_std,
    )

    rollout_dataset = prepare_rollout_training_dataset(
        flat_prompts=flat_prompts,
        flat_responses=flat_responses,
        advantages=advantages,
        raw_rewards=raw_rewards,
        tokenizer=tokenizer,
        policy=policy,
        micro_batch_size=micro_batch_size,
        device=device,
    )

    rollout_metadata = {
        **reward_metadata,
        "num_questions": len(batch_questions),
        "num_responses": len(flat_responses),
    }

    policy.train()
    return rollout_dataset, rollout_metadata

### 6.4.14 GRPO inner train loop

In [ ]:
def train_one_grpo_optimizer_step(
    policy: PreTrainedModel,
    optimizer,
    logical_batch_cpu: Dict[str, torch.Tensor],
    micro_batch_size: int,
    device: str,
    loss_type: str = "grpo_clip",
    clip_eps: float = 0.2,
    length_norm_type: str = "mask_normalize",
    grad_clip_norm: float = 1.0,
):
    """
    对一个 logical batch 执行一次 optimizer.step()。
    logical batch 会在内部拆成多个 micro-batch。
    """
    policy.train()
    optimizer.zero_grad(set_to_none=True)

    total_samples = logical_batch_cpu["input_ids"].size(0)

    if length_norm_type in ["mask_mean", "masked_mean", "sequence_level"]:
        global_normalize_constant = float(total_samples)
    elif length_norm_type in ["mask_normalize", "mask_norm", "token_level"]:
        global_normalize_constant = float(logical_batch_cpu["response_mask"].sum().item())
    else:
        raise ValueError(f"未知 length_norm_type: {length_norm_type}")

    total_loss = 0.0
    total_entropy_num = 0.0
    total_entropy_den = 0.0
    total_ratio_num = 0.0
    total_ratio_den = 0.0
    total_clipped_num = 0.0
    total_clipped_den = 0.0

    for start in range(0, total_samples, micro_batch_size):
        end = min(start + micro_batch_size, total_samples)

        micro = {
            k: v[start:end].to(device)
            for k, v in logical_batch_cpu.items()
        }

        outputs = get_response_log_probs(
            model=policy,
            input_ids=micro["input_ids"],
            labels=micro["labels"],
            attention_mask=micro["attention_mask"],
            return_token_entropy=True,
        )

        current_token_log_probs = outputs["log_probs"]
        token_entropy = outputs["token_entropy"]
        response_mask = micro["response_mask"]
        valid_mask = micro["valid_mask"].bool()
        current_response_mask = response_mask.bool() & valid_mask

        loss, stats = compute_grpo_loss(
            current_token_log_probs=current_token_log_probs,
            response_mask=current_response_mask,
            advantages=micro["advantages"],
            raw_rewards=micro["raw_rewards"],
            old_token_log_probs=micro["old_log_probs"],
            loss_type=loss_type,
            clip_eps=clip_eps,
            length_norm_type=length_norm_type,
            normalize_constant=global_normalize_constant,
        )

        loss.backward()
        total_loss += float(loss.item())

        with torch.no_grad():
            entropy_num = (token_entropy * current_response_mask.float()).sum().item()
            entropy_den = current_response_mask.float().sum().item()
            total_entropy_num += entropy_num
            total_entropy_den += entropy_den

            total_ratio_num += stats["ratio_mean"] * max(entropy_den, 1.0)
            total_ratio_den += max(entropy_den, 1.0)

            total_clipped_num += stats["clip_fraction"] * max(entropy_den, 1.0)
            total_clipped_den += max(entropy_den, 1.0)

    grad_norm = torch.nn.utils.clip_grad_norm_(policy.parameters(), grad_clip_norm)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    metrics = {
        "loss": total_loss,
        "response_entropy": total_entropy_num / max(total_entropy_den, 1.0),
        "ratio_mean": total_ratio_num / max(total_ratio_den, 1.0),
        "clip_fraction": total_clipped_num / max(total_clipped_den, 1.0),
        "grad_norm": float(grad_norm.item()) if torch.is_tensor(grad_norm) else float(grad_norm),
    }
    return metrics

### 6.4.15 对一整批 rollout 数据做 inner loop 训练
这格负责：
- shuffle
- 按 train_batch_size 切 logical batch
- 执行 epochs_per_rollout_batch
- 记录 train metrics

In [ ]:
def train_on_rollout_dataset(
    policy,
    optimizer,
    rollout_dataset_cpu,
    train_batch_size,
    micro_batch_size,
    epochs_per_rollout_batch,
    device,
    global_step_start,
    loss_type="grpo_clip",
    clip_eps=0.2,
    length_norm_type="mask_normalize",
    grad_clip_norm=1.0,
):
    """
    对当前 rollout dataset 做多轮 inner training。
    返回：
      updated_global_step, last_train_metrics
    """
    total_samples = rollout_dataset_cpu["input_ids"].size(0)
    global_step = global_step_start
    last_train_metrics = None

    for _epoch in range(epochs_per_rollout_batch):
        for logical_indices in iterate_logical_batches(
            total_samples=total_samples,
            logical_batch_size=train_batch_size,
            shuffle=True,
        ):
            logical_batch_cpu = slice_tensor_dict(rollout_dataset_cpu, logical_indices)

            global_step += 1

            train_metrics = train_one_grpo_optimizer_step(
                policy=policy,
                optimizer=optimizer,
                logical_batch_cpu=logical_batch_cpu,
                micro_batch_size=micro_batch_size,
                device=device,
                loss_type=loss_type,
                clip_eps=clip_eps,
                length_norm_type=length_norm_type,
                grad_clip_norm=grad_clip_norm,
            )

            wandb.log({
                "train/loss": train_metrics["loss"],
                "train/response_entropy": train_metrics["response_entropy"],
                "train/ratio_mean": train_metrics["ratio_mean"],
                "train/clip_fraction": train_metrics["clip_fraction"],
                "train/grad_norm": train_metrics["grad_norm"],
                "global_step": global_step,
            })

            last_train_metrics = train_metrics

    return global_step, last_train_metrics

### 6.4.16 按频率执行评估

In [ ]:
def evaluate_grpo_if_needed(
    policy,
    vllm_inst,
    eval_params,
    val_samples,
    global_step,
    eval_every_steps,
):
    """
    若达到评测频率，则在验证集上评估一次。
    """
    if global_step % eval_every_steps != 0:
        return None

    print(f"\n[Step {global_step}] 执行评估...")
    policy.eval()
    load_policy_into_vllm_instance(policy, vllm_inst)

    metrics = log_generations(
        vllm_model=vllm_inst,
        sampling_params=eval_params,
        prompts=[s["prompt"] for s in val_samples],
        ground_truths=[s["gold"] for s in val_samples],
        reward_fn=r1_zero_reward_fn,
        step=global_step,
        log_prefix="eval",
    )

    policy.train()
    return metrics

### 6.4.17 完整总装配

In [ ]:
def save_grpo_checkpoint(policy, tokenizer, output_dir, grpo_step, global_step):
    """
    保存 GRPO checkpoint。
    """
    save_path = os.path.join(output_dir, f"grpo_step{grpo_step}_global{global_step}")
    os.makedirs(save_path, exist_ok=True)

    policy.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)

    print(f"Checkpoint saved to {save_path}")
    return save_path

In [ ]:
def run_grpo_experiment(
    model_id,
    train_data_path,
    val_data_path,
    prompt_path,
    output_dir,
    lr=3e-5,
    n_grpo_steps=200,
    train_batch_size=256,
    micro_batch_size=8,
    rollout_batch_size=256,
    group_size=8,
    epochs_per_rollout_batch=1,
    sampling_temperature=1.0,
    sampling_max_tokens=1024,
    sampling_min_tokens=4,
    advantage_eps=1e-6,
    normalize_by_std=True,
    loss_type="grpo_clip",
    clip_eps=0.2,
    length_norm_type="mask_normalize",
    seed=42,
    device="cuda:0",
    vllm_device="cuda:1",
    vllm_gpu_util=0.2,
    max_eval_samples=100,
    eval_every_steps=10,
    wandb_project="cs336-grpo",
    wandb_run_name=None,
):
    config_dict = {
        "model_id": model_id,
        "train_data_path": train_data_path,
        "val_data_path": val_data_path,
        "prompt_path": prompt_path,
        "output_dir": output_dir,
        "lr": lr,
        "n_grpo_steps": n_grpo_steps,
        "train_batch_size": train_batch_size,
        "micro_batch_size": micro_batch_size,
        "rollout_batch_size": rollout_batch_size,
        "group_size": group_size,
        "epochs_per_rollout_batch": epochs_per_rollout_batch,
        "sampling_temperature": sampling_temperature,
        "sampling_max_tokens": sampling_max_tokens,
        "sampling_min_tokens": sampling_min_tokens,
        "advantage_eps": advantage_eps,
        "normalize_by_std": normalize_by_std,
        "loss_type": loss_type,
        "clip_eps": clip_eps,
        "length_norm_type": length_norm_type,
        "seed": seed,
        "device": device,
        "vllm_device": vllm_device,
        "vllm_gpu_util": vllm_gpu_util,
        "max_eval_samples": max_eval_samples,
        "eval_every_steps": eval_every_steps,
        "wandb_project": wandb_project,
        "wandb_run_name": wandb_run_name,
    }

    # 1. 初始化实验
    grad_accum_steps, run, r1_template = init_grpo_experiment(
        train_batch_size=train_batch_size,
        micro_batch_size=micro_batch_size,
        wandb_project=wandb_project,
        wandb_run_name=wandb_run_name,
        prompt_path=prompt_path,
        extra_config=config_dict,
    )

    # 2. 加载模型 / tokenizer / optimizer / vLLM
    tokenizer, policy, optimizer, vllm_inst = load_grpo_models_and_optimizer(
        model_id=model_id,
        device=device,
        vllm_device=vllm_device,
        seed=seed,
        vllm_gpu_util=vllm_gpu_util,
        lr=lr,
    )
    optimizer.zero_grad(set_to_none=True)

    # 3. 数据池与验证集
    question_pool = prepare_question_pool(
        train_data_path=train_data_path,
        prompt_template=r1_template,
    )
    val_samples = prepare_validation_samples(
        val_data_path=val_data_path,
        prompt_template=r1_template,
        max_eval_samples=max_eval_samples,
    )

    eval_params, rollout_params = build_grpo_sampling_params(
        sampling_temperature=sampling_temperature,
        sampling_max_tokens=sampling_max_tokens,
        sampling_min_tokens=sampling_min_tokens,
        group_size=group_size,
    )

    # 4. Step 0 baseline
    run_initial_grpo_evaluation(
        policy=policy,
        vllm_inst=vllm_inst,
        eval_params=eval_params,
        val_samples=val_samples,
    )

    # 5. 外层循环
    global_step = 0

    for grpo_step in range(1, n_grpo_steps + 1):
        print(f"\n{'=' * 24} 开始 GRPO Step {grpo_step} / {n_grpo_steps} {'=' * 24}")

        # A. 采样问题
        batch_questions = sample_rollout_questions(
            question_pool=question_pool,
            rollout_batch_size=rollout_batch_size,
            group_size=group_size,
        )

        # B. rollout + reward + advantage + old_log_probs
        rollout_dataset_cpu, rollout_metadata = run_one_grpo_rollout(
            policy=policy,
            vllm_inst=vllm_inst,
            tokenizer=tokenizer,
            batch_questions=batch_questions,
            rollout_params=rollout_params,
            micro_batch_size=micro_batch_size,
            device=device,
            reward_fn=r1_zero_reward_fn,
            group_size=group_size,
            advantage_eps=advantage_eps,
            normalize_by_std=normalize_by_std,
        )

        wandb.log({
            "rollout/mean_reward": rollout_metadata["mean_reward"],
            "rollout/std_reward": rollout_metadata["std_reward"],
            "rollout/max_reward": rollout_metadata["max_reward"],
            "rollout/min_reward": rollout_metadata["min_reward"],
            "rollout/mean_advantage": rollout_metadata["mean_advantage"],
            "rollout/num_questions": rollout_metadata["num_questions"],
            "rollout/num_responses": rollout_metadata["num_responses"],
            "grpo_step": grpo_step,
        })

        print(
            f">> rollout mean_reward={rollout_metadata['mean_reward']:.4f}, "
            f"max_reward={rollout_metadata['max_reward']:.4f}, "
            f"mean_advantage={rollout_metadata['mean_advantage']:.4f}"
        )

        # C. inner train steps
        global_step, last_train_metrics = train_on_rollout_dataset(
            policy=policy,
            optimizer=optimizer,
            rollout_dataset_cpu=rollout_dataset_cpu,
            train_batch_size=train_batch_size,
            micro_batch_size=micro_batch_size,
            epochs_per_rollout_batch=epochs_per_rollout_batch,
            device=device,
            global_step_start=global_step,
            loss_type=loss_type,
            clip_eps=clip_eps,
            length_norm_type=length_norm_type,
            grad_clip_norm=1.0,
        )

        # D. 周期性评估
        evaluate_grpo_if_needed(
            policy=policy,
            vllm_inst=vllm_inst,
            eval_params=eval_params,
            val_samples=val_samples,
            global_step=global_step,
            eval_every_steps=eval_every_steps,
        )

        # E. 每个外层 step 保存一次
        save_grpo_checkpoint(
            policy=policy,
            tokenizer=tokenizer,
            output_dir=output_dir,
            grpo_step=grpo_step,
            global_step=global_step,
        )

        torch.cuda.empty_cache()

    print("\nGRPO 训练全部完成")
    wandb.finish()

    return {
        "policy": policy,
        "tokenizer": tokenizer,
        "vllm_inst": vllm_inst,
        "global_step": global_step,
    }

In [ ]:
run_grpo_experiment(
    model_id="model/Qwen2.5-Math-1.5B",
    train_data_path="data/gsm8k/train_sft_reason_gsm8k_raw.jsonl",
    val_data_path="data/gsm8k/test.jsonl",
    prompt_path="cs336_alignment/prompts/r1_zero.prompt",
    output_dir="result/grpo_checkpoints",

    lr=3e-5,
    n_grpo_steps=50,

    train_batch_size=256,
    micro_batch_size=8,

    rollout_batch_size=256,
    group_size=8,
    epochs_per_rollout_batch=1,

    sampling_temperature=1.0,
    sampling_max_tokens=1024,
    sampling_min_tokens=4,

    advantage_eps=1e-6,
    normalize_by_std=True,

    loss_type="grpo_clip",
    clip_eps=0.2,
    length_norm_type="mask_normalize",

    seed=42,
    device="cuda:0",
    vllm_device="cuda:1",
    vllm_gpu_util=0.2,

    max_eval_samples=100,
    eval_every_steps=10,

    wandb_project="cs336-grpo",
    wandb_run_name="grpo-demo",
)

**第一轮：只跑数学核心**

先跑这些 cell，看 shape 和 loss 行为：
- Cell 11
- Cell 12
- Cell 17

这轮目标只有一个：
搞懂 group advantage、clip surrogate、length norm 三者是怎么在代码里拼起来的。

**第二轮：再跑 rollout 数据链**
- Cell 13
- Cell 14
- Cell 15
- Cell 24

这轮目标是：
搞懂从 vLLM 输出，到 raw reward，到 advantage，到 old_log_probs，是怎么变成训练 batch 的。

**第三轮：最后跑训练闭环**
- Cell 25
- Cell 26
- Cell 27
- Cell 28
- Cell 29
- Cell 30

这轮会真正看到：
- 外层 rollout
- 内层多次 gradient update
- eval / save
- 完整 GRPO loop